# Notebook para experimentação de tipos de filtragens 
Autor: Suzane Carol de Lima  
Entrega: 05/03/26
## Objetivo:
Notebook de criação e experimentação de filtros em shapefile de pontos

## Importações de bibliotecas

In [3]:
import geopandas as gpd
from geopandas import GeoDataFrame
from sklearn.cluster import DBSCAN
import numpy as np

# Funções necessárias

In [4]:
def filtrar_pontos_proximos(
    arquivo_pontos: str,
    arquivo_linha: str,
    arquivo_saida: str,
    buffer_metros: float = 10,
    epsg: str = 'EPSG:31984'
) -> GeoDataFrame:
    """
    Filtra pontos que estão a uma distância máxima especificada de uma linha geográfica.

    Parâmetros:
        arquivo_pontos (str): Caminho para o arquivo de pontos (shapefile).
        arquivo_linha (str): Caminho para o arquivo da linha (shapefile).
        arquivo_saida (str): Caminho para salvar o shapefile de saída com os pontos filtrados.
        buffer_metros (float, opcional): Distância máxima em metros para considerar o ponto próximo da linha. Padrão é 10.
        epsg (str, opcional): Código EPSG do sistema de referência. Padrão é 'EPSG:31984'.

    Retorno:
        GeoDataFrame: GeoDataFrame contendo apenas os pontos dentro da distância especificada da linha.
    """
    pontos = gpd.read_file(arquivo_pontos)
    linha = gpd.read_file(arquivo_linha)
    pontos = pontos.to_crs(epsg)
    linha = linha.to_crs(epsg)
    linha_geom = linha.unary_union
    pontos['dist'] = pontos.geometry.distance(linha_geom)
    pontos_filtrados = pontos[pontos['dist'] <= buffer_metros].drop(columns='dist').reset_index(drop=True)
    pontos_filtrados.to_file(arquivo_saida, driver='ESRI Shapefile', encoding='utf-8')
    return pontos_filtrados

def selecionar_pontos_coordenadas_validas(
    arquivo_entrada: str,
    arquivo_saida: str
) -> GeoDataFrame:
    """
    Seleciona pontos com coordenadas válidas (valores diferentes de zero).

    Parâmetros:
        arquivo_entrada (str): Caminho para o shapefile de entrada.
        arquivo_saida (str): Caminho para salvar o shapefile de saída com os pontos filtrados.

    Retorno:
        GeoDataFrame: GeoDataFrame contendo apenas os pontos com coordenadas válidas.
    """
    gdf = gpd.read_file(arquivo_entrada)
    gdf_filtrado = gdf[(gdf['north'] != 0) & (gdf['east'] != 0)]
    gdf_filtrado.to_file(arquivo_saida)
    return gdf_filtrado

def selecionar_pontos_por_dept(
    arquivo_entrada: str,
    arquivo_saida: str,
    valor_minimo_dept: float
) -> GeoDataFrame:
    """
    Seleciona pontos com valor de profundidade ('depth') maior que um valor mínimo.

    Parâmetros:
        arquivo_entrada (str): Caminho para o shapefile de entrada.
        arquivo_saida (str): Caminho para salvar o shapefile de saída com os pontos filtrados.
        valor_minimo_dept (float): Valor mínimo de profundidade para filtrar os pontos.

    Retorno:
        GeoDataFrame: GeoDataFrame contendo apenas os pontos com profundidade maior que o valor mínimo.
    """
    gdf = gpd.read_file(arquivo_entrada)
    gdf_filtrado = gdf[gdf['depth'] > valor_minimo_dept]
    gdf_filtrado.to_file(arquivo_saida)
    return gdf_filtrado

def remover_outliers_dbscan(
    arquivo_entrada: str,
    arquivo_saida: str,
    eps: float = 10,
    min_amostras: int = 5
) -> GeoDataFrame:
    """
    Remove outliers espaciais de pontos usando o algoritmo DBSCAN.

    Parâmetros:
        arquivo_entrada (str): Caminho para o shapefile de entrada.
        arquivo_saida (str): Caminho para salvar o shapefile de saída sem outliers.
        eps (float, opcional): Distância máxima entre pontos para serem considerados vizinhos. Padrão é 10.
        min_amostras (int, opcional): Número mínimo de amostras por cluster. Padrão é 5.

    Retorno:
        GeoDataFrame: GeoDataFrame com pontos filtrados, sem outliers.
    """
    gdf = gpd.read_file(arquivo_entrada)
    coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    db = DBSCAN(eps=eps, min_samples=min_amostras).fit(coords)
    gdf['cluster'] = db.labels_
    gdf_filtrado = gdf[gdf['cluster'] != -1]
    gdf_filtrado = gdf_filtrado.drop(columns=['cluster'])
    gdf_filtrado.to_file(arquivo_saida)
    return gdf_filtrado

def selecionar_pontos_por_distancia(
    entrada_shp: str,
    saida_shp: str,
    distancia_minima: float = 50,
    epsg: int = 31984
) -> GeoDataFrame:
    """
    Seleciona pontos garantindo uma distância mínima entre eles.

    Parâmetros:
        entrada_shp (str): Caminho para o shapefile de entrada.
        saida_shp (str): Caminho para salvar o shapefile filtrado.
        distancia_minima (float, opcional): Distância mínima entre pontos selecionados. Padrão é 50.
        epsg (int, opcional): Código EPSG do sistema de referência. Padrão é 31984.

    Retorno:
        GeoDataFrame: GeoDataFrame com os pontos selecionados respeitando a distância mínima.
    """
    gdf = gpd.read_file(entrada_shp)
    gdf = gdf.to_crs(epsg=epsg)
    selecionados = []
    for idx, linha in gdf.iterrows():
        if not selecionados:
            selecionados.append(linha)
        else:
            menor_dist = min([linha.geometry.distance(sel.geometry) for sel in selecionados])
            if menor_dist >= distancia_minima:
                selecionados.append(linha)
    gdf_selecionados = gpd.GeoDataFrame(selecionados, crs=gdf.crs)
    gdf_selecionados.to_file(saida_shp)
    return gdf_selecionados




# Filtrar pontos mais próximos da estrutura (shapefile) de referência

In [12]:
filtrar_pontos_proximos('../../../SHAPEFILES/85520-6000685899 - Apollo Z/shape_rota_ROV.shp', '../../../SHAPEFILES/OLEODUTO.shp', '../../../SHAPEFILES/85520-6000685899 - Apollo Z/pontos_filtrados.shp', buffer_metros=10)

/tmp/ipykernel_1735878/78708891.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  linha_geom = linha.unary_union


,nome_video,frame_id,time,north,east,depth,geometry
0,2024-03-29_181802_Ch4_00,Frame_599_2024-03-29_181802_Ch4_00_00_19,00:19,9456943.83,803963.00,16.00,POINT (803963 9456943.83)
1,2024-03-29_181802_Ch4_00,Frame_1198_2024-03-29_181802_Ch4_00_00_39,00:39,9456943.71,803962.96,16.09,POINT (803962.96 9456943.71)
2,2024-03-29_181802_Ch4_00,Frame_1797_2024-03-29_181802_Ch4_00_00_59,00:59,9456943.68,803963.04,16.16,POINT (803963.04 9456943.68)
3,2024-03-29_181802_Ch4_00,Frame_2995_2024-03-29_181802_Ch4_00_01_39,01:39,9456943.78,803963.04,16.02,POINT (803963.04 9456943.78)
4,2024-03-29_181802_Ch4_00,Frame_4193_2024-03-29_181802_Ch4_00_02_19,02:19,9456944.59,803963.84,16.04,POINT (803963.84 9456944.59)
...,...,...,...,...,...,...,...
2268,6000685899_2024-03-30_041054_Ch4_32,Frame_21534_6000685899_2024-03-30_041054_Ch4_3...,12:18,9456014.88,804315.91,-0.12,POINT (804315.91 9456014.88)
2269,6000685899_2024-03-30_041054_Ch4_32,Frame_22116_6000685899_2024-03-30_041054_Ch4_3...,12:38,9456014.73,804316.75,-0.12,POINT (804316.75 9456014.73)
2270,6000685899_2024-03-30_041054_Ch4_32,Frame_22698_6000685899_2024-03-30_041054_Ch4_3...,12:58,9456014.47,804316.99,-0.12,POINT (804316.99 9456014.47)
2271,6000685899_2024-03-30_041054_Ch4_32,Frame_23280_6000685899_2024-03-30_041054_Ch4_3...,13:18,9456014.08,804317.63,-0.12,POINT (804317.63 9456014.08)


# Filtrar pontos pela profundidade

In [13]:
selecionar_pontos_por_dept('../../../SHAPEFILES/85520-6000685899 - Apollo Z/shape_rota_ROV.shp', '../../../SHAPEFILES/85520-6000685899 - Apollo Z/saida_filtrada_profundidade.shp', 1)

,nome_video,frame_id,time,north,east,depth,geometry
0,2024-03-29_181802_Ch4_00,Frame_00_2024-03-29_181802_Ch4_00_00_00,00:00,8456943.88,803963.09,16.03,POINT (803963.09 8456943.88)
1,2024-03-29_181802_Ch4_00,Frame_599_2024-03-29_181802_Ch4_00_00_19,00:19,9456943.83,803963.00,16.00,POINT (803963 9456943.83)
2,2024-03-29_181802_Ch4_00,Frame_1198_2024-03-29_181802_Ch4_00_00_39,00:39,9456943.71,803962.96,16.09,POINT (803962.96 9456943.71)
3,2024-03-29_181802_Ch4_00,Frame_1797_2024-03-29_181802_Ch4_00_00_59,00:59,9456943.68,803963.04,16.16,POINT (803963.04 9456943.68)
4,2024-03-29_181802_Ch4_00,Frame_2396_2024-03-29_181802_Ch4_00_01_19,01:19,9456943.58,803952.97,16.08,POINT (803952.97 9456943.58)
...,...,...,...,...,...,...,...
2432,6000685899_2024-03-30_041054_Ch4_32,Frame_12804_6000685899_2024-03-30_041054_Ch4_3...,07:19,9456009.86,804317.96,9.47,POINT (804317.96 9456009.86)
2433,6000685899_2024-03-30_041054_Ch4_32,Frame_13386_6000685899_2024-03-30_041054_Ch4_3...,07:39,9456009.88,804317.87,9.43,POINT (804317.87 9456009.88)
2434,6000685899_2024-03-30_041054_Ch4_32,Frame_13968_6000685899_2024-03-30_041054_Ch4_3...,07:59,9456009.67,804317.71,8.53,POINT (804317.71 9456009.67)
2435,6000685899_2024-03-30_041054_Ch4_32,Frame_14550_6000685899_2024-03-30_041054_Ch4_3...,08:19,9456010.49,804318.00,6.71,POINT (804318 9456010.49)


# Filtrar pontos com coordenadas validas (diferente de 0)

In [14]:
selecionar_pontos_coordenadas_validas('../../../SHAPEFILES/85520-6000685899 - Apollo Z/shape_rota_ROV.shp', '../../../SHAPEFILES/85520-6000685899 - Apollo Z/saida_filtrada_coordeanda.shp')

,nome_video,frame_id,time,north,east,depth,geometry
0,2024-03-29_181802_Ch4_00,Frame_00_2024-03-29_181802_Ch4_00_00_00,00:00,8456943.88,803963.09,16.03,POINT (803963.09 8456943.88)
1,2024-03-29_181802_Ch4_00,Frame_599_2024-03-29_181802_Ch4_00_00_19,00:19,9456943.83,803963.00,16.00,POINT (803963 9456943.83)
2,2024-03-29_181802_Ch4_00,Frame_1198_2024-03-29_181802_Ch4_00_00_39,00:39,9456943.71,803962.96,16.09,POINT (803962.96 9456943.71)
3,2024-03-29_181802_Ch4_00,Frame_1797_2024-03-29_181802_Ch4_00_00_59,00:59,9456943.68,803963.04,16.16,POINT (803963.04 9456943.68)
4,2024-03-29_181802_Ch4_00,Frame_2396_2024-03-29_181802_Ch4_00_01_19,01:19,9456943.58,803952.97,16.08,POINT (803952.97 9456943.58)
...,...,...,...,...,...,...,...
2447,6000685899_2024-03-30_041054_Ch4_32,Frame_21534_6000685899_2024-03-30_041054_Ch4_3...,12:18,9456014.88,804315.91,-0.12,POINT (804315.91 9456014.88)
2448,6000685899_2024-03-30_041054_Ch4_32,Frame_22116_6000685899_2024-03-30_041054_Ch4_3...,12:38,9456014.73,804316.75,-0.12,POINT (804316.75 9456014.73)
2449,6000685899_2024-03-30_041054_Ch4_32,Frame_22698_6000685899_2024-03-30_041054_Ch4_3...,12:58,9456014.47,804316.99,-0.12,POINT (804316.99 9456014.47)
2450,6000685899_2024-03-30_041054_Ch4_32,Frame_23280_6000685899_2024-03-30_041054_Ch4_3...,13:18,9456014.08,804317.63,-0.12,POINT (804317.63 9456014.08)


#  Filtrar pontos com DBSCAN  
## Como funciona o DBSCAN  
DBSCAN é um método que agrupa pontos próximos, formando “grupos” de dados que estão juntos e separando pontos que estão isolados, chamando esses de “outliers”. Ele funciona como se você desenhasse círculos ao redor de cada ponto; se muitos círculos se sobrepõem, eles formam um grupo. Pontos que ficam sozinhos, sem ninguém por perto, são descartados. Ou seja, ele acha as regiões mais “populosas” e ignora quem está muito afastado.  

![Exemplo de funcionamento do DBSCAN](exemplo-DBSCAN/DBSCAN.png)

No nosso caso, ele olha para cada ponto do arquivo de pontos e verifica se ele está perto de outros pontos, considerando uma distância definada (eps = 10 metros). Se um ponto está perto de pelo menos uma quantidade mínima de vizinhos (min_amostras = 5), ele faz parte de um grupo (cluster). Se não, ele é considerado um ponto estranho, fora do padrão (outlier), e é removido do resultado. No final, só ficam os pontos que estão próximos de outros, formando grupos, e os isolados são descartados.

In [15]:
remover_outliers_dbscan("../../../SHAPEFILES/85520-6000685899 - Apollo Z/shape_rota_ROV.shp", "../../../SHAPEFILES/85520-6000685899 - Apollo Z/shapefile_sem_outliers_DBSCAN_10.shp", eps=10, min_amostras=5)

,nome_video,frame_id,time,north,east,depth,geometry
1,2024-03-29_181802_Ch4_00,Frame_599_2024-03-29_181802_Ch4_00_00_19,00:19,9456943.83,803963.00,16.00,POINT (803963 9456943.83)
2,2024-03-29_181802_Ch4_00,Frame_1198_2024-03-29_181802_Ch4_00_00_39,00:39,9456943.71,803962.96,16.09,POINT (803962.96 9456943.71)
3,2024-03-29_181802_Ch4_00,Frame_1797_2024-03-29_181802_Ch4_00_00_59,00:59,9456943.68,803963.04,16.16,POINT (803963.04 9456943.68)
4,2024-03-29_181802_Ch4_00,Frame_2396_2024-03-29_181802_Ch4_00_01_19,01:19,9456943.58,803952.97,16.08,POINT (803952.97 9456943.58)
5,2024-03-29_181802_Ch4_00,Frame_2995_2024-03-29_181802_Ch4_00_01_39,01:39,9456943.78,803963.04,16.02,POINT (803963.04 9456943.78)
...,...,...,...,...,...,...,...
2447,6000685899_2024-03-30_041054_Ch4_32,Frame_21534_6000685899_2024-03-30_041054_Ch4_3...,12:18,9456014.88,804315.91,-0.12,POINT (804315.91 9456014.88)
2448,6000685899_2024-03-30_041054_Ch4_32,Frame_22116_6000685899_2024-03-30_041054_Ch4_3...,12:38,9456014.73,804316.75,-0.12,POINT (804316.75 9456014.73)
2449,6000685899_2024-03-30_041054_Ch4_32,Frame_22698_6000685899_2024-03-30_041054_Ch4_3...,12:58,9456014.47,804316.99,-0.12,POINT (804316.99 9456014.47)
2450,6000685899_2024-03-30_041054_Ch4_32,Frame_23280_6000685899_2024-03-30_041054_Ch4_3...,13:18,9456014.08,804317.63,-0.12,POINT (804317.63 9456014.08)


# Selecionar pontos a cada 50 metros

In [17]:
selecionar_pontos_por_distancia(
    '../../../SHAPEFILES/85515-6000685758 - Apollo Z/pontos_filtrados.shp',
    '../../../SHAPEFILES/85515-6000685758 - Apollo Z/pontos_filtrados_50metros.shp'
)

,nome_video,frame_id,time,north,east,depth,geometry
0,6000685758_PAG-03_2024-03-23_023756_Ch4_52,Frame_18569_6000685758_PAG-03_2024-03-23_02375...,10:18,9455981.78,804315.14,16.13,POINT (804315.14 9455981.78)
120,6000685758_PAG-03_2024-03-23_043738_Ch4_60,Frame_599_6000685758_PAG-03_2024-03-23_043738_...,00:19,9456025.55,804275.03,17.99,POINT (804275.03 9456025.55)
393,6000685758_PAG-03_2024-03-23_100648_Ch4_82,Frame_5391_6000685758_PAG-03_2024-03-23_100648...,02:59,9456075.55,804259.39,17.48,POINT (804259.39 9456075.55)
413,6000685758_PAG-03_2024-03-23_100648_Ch4_82,Frame_17371_6000685758_PAG-03_2024-03-23_10064...,09:39,9456123.28,804237.02,18.53,POINT (804237.02 9456123.28)
486,6000685758_PAG-03_2024-03-23_103643_Ch4_84,Frame_8386_6000685758_PAG-03_2024-03-23_103643...,04:39,9456174.07,804222.02,18.58,POINT (804222.02 9456174.07)
530,6000685758_PAG-03_2024-03-23_105140_Ch4_85,Frame_7188_6000685758_PAG-03_2024-03-23_105140...,03:59,9456222.81,804203.65,18.11,POINT (804203.65 9456222.81)
578,6000685758_PAG-03_2024-03-23_110638_Ch4_86,Frame_8386_6000685758_PAG-03_2024-03-23_110638...,04:39,9456275.45,804189.14,16.01,POINT (804189.14 9456275.45)
614,6000685758_PAG-03_2024-03-23_112136_Ch4_87,Frame_4193_6000685758_PAG-03_2024-03-23_112136...,02:19,9456326.09,804174.22,16.93,POINT (804174.22 9456326.09)
718,6000685758_PAG-03_2024-03-23_115131_Ch4_89,Frame_18569_6000685758_PAG-03_2024-03-23_11513...,10:18,9456377.63,804160.64,16.11,POINT (804160.64 9456377.63)
731,6000685758_PAG-03_2024-03-23_115131_Ch4_89,Frame_26356_6000685758_PAG-03_2024-03-23_11513...,14:38,9456426.10,804148.20,15.87,POINT (804148.2 9456426.1)
